# Check save and load data.

In [1]:
%run -i ~/project/preambles
%run -i ~/project/helper_functions
%run -i ~/project/fitting_functions-Copy1

cpu


In [2]:
# # Global parameters:
number_of_cycles = 500 # how many passes through the training data we go through
number_of_groups = 100 # divide the data set into smaller ones, to make fitting easier.
steps_per_batch = 5
dims = 2  # 2D spatial
head = 120 # how many locations to consider in the real data set.

In [3]:
gene_list = ["Inha", "Esr1"]
puck_list = ['Puck_230517_39'] # the largest puck
# gene_list = [ "Inha", "Inhba", "Inhbb", "Fst", "Esr1", "Esr2", "Pgr", "Ar", "Cyp19a1", 
#     "Cyp17a1", "Cyp11a1", "Lhcgr", "Parm1", "Akr1c18", "Fshr", "Star", "Ptgfr", 
#     "Sfrp4", "Acvr1", "Acvr2a", "Acvr2b", "Ghr", "Lhb", "Cga"]

# puck_list = ['Puck_230223_01', 'Puck_230406_01', 'Puck_230406_06', 'Puck_230406_08', 
#              'Puck_230517_37', 'Puck_230517_38', 'Puck_230517_39', 'Puck_230913_07', 
#              'Puck_240108_20', 'Puck_240108_24', 'Puck_240108_25', 'Puck_240108_26', 
#              'A0029_047', 'A0029_043', 'Puck_230807_27', 'Puck_240129_36', 'Puck_240129_37', 
#              'A0029_042', 'A0029_036', 'PM104_004', 'Puck_240108_10', 'Puck_240108_11', 
#              'Puck_230807_04', 'Puck_230714_28', 'Puck_230714_23']
# puck_list = 'all'
adata, X,Y,df, gene_list=load_data(gene_list=gene_list, head=head, puck_list = puck_list)

In [4]:
filepath = "optimized_marginal_params.csv"
def marginal_fitting_codes(head, gene_list):
    adata, X,Y,df, gene_list=load_data(gene_list=gene_list, head=head, puck_list = puck_list)
    optimized_marginal_params = optimize_marginal_parameters(X, Y, number_of_groups,  number_of_cycles, steps_per_batch)
    df_to_save = pd.DataFrame(optimized_marginal_params) # list to df
    df_to_save.to_csv(filepath, index=False)
    return None

In [ ]:
# Combine the marginal fitted parameters into a single csv file
indices = [101, 102, 103, 104] # provide the correct indices in the correct order
filepaths = [os.path.expanduser(f"~/project/python_processed_data/fitted_parameters_{i}.csv") for i in indices]
df = pd.concat([pd.read_csv(filepath) for filepath in filepaths])
first_index = indices[0]
filepath = os.path.expanduser(f"~/project/python_processed_data/fitted_parameters_{first_index}.csv")
df.to_csv(filepath, index=False)

In [5]:
# this block runs the loading and the optimisation steps 
def cross_fitting_codes(head, gene_list):
    optimized_marginal_params = pd.read_csv(filepath).values.tolist()# df to list
    estimated_params_df = pd.DataFrame()
    alpha_matrix, nu_matrix, sigma_matrix = optimize_cross_parameters(optimized_marginal_params,X,Y,number_of_groups,number_of_cycles,steps_per_batch)
    estimated_params_df = pd.concat([estimated_params_df, store_as_df(alpha_matrix, nu_matrix, sigma_matrix) ], ignore_index=True)
    return estimated_params_df

In [6]:
# Memory & Time tracking function # Load the memory profiler magic
%load_ext memory_profiler
import time

In [7]:
%%memit 
start_time = time.time()
marginal_fitting_codes(head,gene_list)
elapsed_time =(time.time() - start_time)/3600
print(f"Time taken: {elapsed_time:.6f} hours")

Time taken: 0.005956 hours
peak memory: 3497.80 MiB, increment: 1488.56 MiB


In [8]:
%run -i ~/project/fitting_functions-Copy1

In [9]:
%%memit 
start_time = time.time()
estimated_params_df = cross_fitting_codes(head, gene_list)
elapsed_time =(time.time() - start_time)/3600
print(f"Time taken: {elapsed_time:.6f} hours")
estimated_params_df

Saving cross fitting checkpoint at epoch 0
Saving cross fitting checkpoint at epoch 10
Saving cross fitting checkpoint at epoch 20
Saving cross fitting checkpoint at epoch 30
Saving cross fitting checkpoint at epoch 40
Saving cross fitting checkpoint before timeout at epoch 45
Time taken: 0.004228 hours
peak memory: 2205.50 MiB, increment: 8.12 MiB


In [10]:
%%memit 
start_time = time.time()
estimated_params_df = cross_fitting_codes(head, gene_list)
elapsed_time =(time.time() - start_time)/3600
print(f"Time taken: {elapsed_time:.6f} hours")
estimated_params_df

Resuming cross fitting from epoch 45
Loaded Delta_A: 0.900053173521796
Loaded Delta_B: 0.9000000000325828
Loaded rho_A: 0.09994682647398004
Loaded rho_B: 0.09999999996742676
Loaded rho_V: -0.09999999979939944
Loaded W: tensor([2.2204e-16, 2.2204e-16], dtype=torch.float64, requires_grad=True)
Loaded best_loss: 8864032549.192883
Saving cross fitting checkpoint at epoch 50
Saving cross fitting checkpoint at epoch 60
Saving cross fitting checkpoint at epoch 70
Saving cross fitting checkpoint at epoch 80
Saving cross fitting checkpoint at epoch 90
Saving cross fitting checkpoint before timeout at epoch 90
Time taken: 0.004216 hours
peak memory: 2206.01 MiB, increment: 0.50 MiB


In [11]:
%%memit 
start_time = time.time()
estimated_params_df = cross_fitting_codes(head, gene_list)
elapsed_time =(time.time() - start_time)/3600
print(f"Time taken: {elapsed_time:.6f} hours")
estimated_params_df

Resuming cross fitting from epoch 90
Loaded Delta_A: 0.9001063490367388
Loaded Delta_B: 0.9000000000651657
Loaded rho_A: 0.09989365095481426
Loaded rho_B: 0.0999999999348535
Loaded rho_V: -0.09999999959880866
Loaded W: tensor([2.2204e-16, 2.2204e-16], dtype=torch.float64, requires_grad=True)
Loaded best_loss: 8864032549.192883
Saving cross fitting checkpoint at epoch 90
Saving cross fitting checkpoint at epoch 100
Saving cross fitting checkpoint at epoch 110
Saving cross fitting checkpoint at epoch 120
Saving cross fitting checkpoint at epoch 130
Saving cross fitting checkpoint before timeout at epoch 135
Time taken: 0.004215 hours
peak memory: 2206.03 MiB, increment: 0.02 MiB


In [12]:
%%memit 
start_time = time.time()
estimated_params_df = cross_fitting_codes(head, gene_list)
elapsed_time =(time.time() - start_time)/3600
print(f"Time taken: {elapsed_time:.6f} hours")
estimated_params_df

Resuming cross fitting from epoch 135
Loaded Delta_A: 0.9001595265502238
Loaded Delta_B: 0.9000000000977485
Loaded rho_A: 0.09984047343710643
Loaded rho_B: 0.09999999990228282
Loaded rho_V: -0.09999999939822767
Loaded W: tensor([2.2204e-16, 2.2204e-16], dtype=torch.float64, requires_grad=True)
Loaded best_loss: 8864032549.192883
Saving cross fitting checkpoint at epoch 140
Saving cross fitting checkpoint at epoch 150
Saving cross fitting checkpoint at epoch 160
Saving cross fitting checkpoint at epoch 170
Saving cross fitting checkpoint at epoch 180
Saving cross fitting checkpoint before timeout at epoch 180
Time taken: 0.004217 hours
peak memory: 2206.03 MiB, increment: 0.00 MiB


In [13]:
%%memit 
start_time = time.time()
estimated_params_df = cross_fitting_codes(head, gene_list)
elapsed_time =(time.time() - start_time)/3600
print(f"Time taken: {elapsed_time:.6f} hours")
estimated_params_df

Resuming cross fitting from epoch 180
Loaded Delta_A: 0.900212706062227
Loaded Delta_B: 0.9000000001303313
Loaded rho_A: 0.09978729392088104
Loaded rho_B: 0.09999999986971277
Loaded rho_V: -0.09999999919765641
Loaded W: tensor([2.2204e-16, 2.2204e-16], dtype=torch.float64, requires_grad=True)
Loaded best_loss: 8864032549.192883
Saving cross fitting checkpoint at epoch 180
Saving cross fitting checkpoint at epoch 190
Saving cross fitting checkpoint at epoch 200
Saving cross fitting checkpoint at epoch 210
Saving cross fitting checkpoint at epoch 220
Saving cross fitting checkpoint before timeout at epoch 225
Time taken: 0.004214 hours
peak memory: 2206.03 MiB, increment: 0.00 MiB


### Memory Profiling ends

### At this stage, we have obtained our data.

In [14]:
# estimated_params_df = pd.DataFrame()
# distance_K_df = pd.DataFrame()
# for _ in range(1):
#     try:
#         torch.autograd.set_detect_anomaly(True)
#         optimized_marginal_params = optimize_marginal_parameters(X, Y, number_of_groups,  number_of_cycles, steps_per_batch)
#         print(optimized_marginal_params)
#         alpha_matrix, nu_matrix, sigma_matrix = optimize_cross_parameters(optimized_marginal_params,X,Y,number_of_groups,number_of_cycles,steps_per_batch)
#         # estimated_K = compute_matern_covariance(alpha_matrix, nu_matrix, sigma_matrix, X)
#         # distance_K = torch.norm(true_K - estimated_K)**2 / torch.norm(true_K)**2
#         # distance_K_df = pd.concat([distance_K_df, pd.DataFrame([distance_K.item()])] ,ignore_index=True)
#         estimated_params_df = pd.concat([estimated_params_df, store_as_df(alpha_matrix, nu_matrix, sigma_matrix) ], ignore_index=True)
#     except Exception as e:
#         print(e)
#         continue
# estimated_params_df

In [15]:
# estimated_params_df = pd.DataFrame()
# distance_K_df = pd.DataFrame()
# for _ in range(number_of_simulations):
#     try:
#         optimized_marginal_params = optimize_marginal_parameters(X, Y, number_of_groups,  number_of_cycles, steps_per_batch)
#         alpha_matrix, nu_matrix, sigma_matrix = optimize_cross_parameters(optimized_marginal_params,X,Y,number_of_groups,number_of_cycles,steps_per_batch)
#         estimated_K = compute_matern_covariance(alpha_matrix, nu_matrix, sigma_matrix, X)
#         estimated_params_df = pd.concat([estimated_params_df, store_as_df(alpha_matrix, nu_matrix, sigma_matrix) ], ignore_index=True)
#     except Exception as e:
#         print(e)
#         continue
# estimated_params_df

In [16]:
# import matplotlib.pyplot as plt
# import math

# # Get the list of columns
# columns = estimated_params_df.columns

# # Calculate the number of rows and columns for subplots
# n_params = len(columns)
# n_cols = 3  # Fixed number of columns for layout
# n_rows = math.ceil(n_params / n_cols)  # Calculate required rows based on the number of parameters

# plt.figure(figsize=(15, 2.5 * n_rows))  # Adjust height based on number of rows
# for i, col in enumerate(columns):
#     plt.subplot(n_rows, n_cols, i + 1)
    
#     # Plot the histogram of estimates
#     plt.hist(estimated_params_df[col], bins=30, color='skyblue', edgecolor='black')
    
#     # Plot the vertical line for the true value
#     # plt.axvline(x=ground_truth_df[col].iloc[0], color='red', linestyle='--', linewidth=2)
    
#     plt.title(f'{col} Distribution')
#     plt.xlabel(f'{col}')
#     plt.ylabel('Frequency')
#     plt.grid(True)

# plt.tight_layout()
# plt.show()

Save the computed parameters

In [17]:
# # Concatenate the two DataFrames along the columns (axis=1)
# # combined_df = pd.concat([estimated_params_df, distance_K_df], axis=1)
# # # Define the file path using f-string and expand the home directory
# file_path = os.path.expanduser(f'~/project/python_processed_data/estimated_parameters_4.csv')
# # # Save the combined DataFrame to a CSV file
# # combined_df.to_csv(file_path, index=False)
# # combined_df
# estimated_params_df.to_csv(file_path, index=False)